#BRONZE LAYER - ADVENTUREWORKS RETAIL LAKEHOUSE PROJECT

##  Step 1: Setup Paths

In [0]:
base_path = "/Volumes/adventureworks-lakehouse-project/default/adventureworks/"
bronze_path = "/Volumes/adventureworks-lakehouse-project/default/adventureworks/bronze/"

## Step 2: Import Libraries 

In [0]:
from pyspark.sql.functions import current_timestamp, lit

## Step 3: Check Files

In [0]:
display(dbutils.fs.ls(base_path))

## Step 4: Load Sales Data

In [0]:
sales_2015 = spark.read.csv(base_path + "AdventureWorks_Sales_2015.csv", header=True, inferSchema=True)

sales_2016 = spark.read.csv(base_path + "AdventureWorks_Sales_2016.csv", header=True, inferSchema=True)

sales_2017 = spark.read.csv(base_path + "AdventureWorks_Sales_2017.csv", header=True, inferSchema=True)

## Step 5: Combine Sales

In [0]:
sales = sales_2015.unionByName(sales_2016).unionByName(sales_2017)

## Step 6: Load Dimension & Other Tables

In [0]:
customers = spark.read.csv(base_path + "AdventureWorks_Customers.csv", header=True, inferSchema=True)

products = spark.read.csv(base_path + "AdventureWorks_Products.csv", header=True, inferSchema=True)

categories = spark.read.csv(base_path + "AdventureWorks_Product_Categories.csv", header=True, inferSchema=True)

subcategories = spark.read.csv(base_path + "AdventureWorks_Product_Subcategories.csv", header=True, inferSchema=True)

returns = spark.read.csv(base_path + "AdventureWorks_Returns.csv", header=True, inferSchema=True)

territories = spark.read.csv(base_path + "AdventureWorks_Territories.csv", header=True, inferSchema=True)

calendar = spark.read.csv(base_path + "AdventureWorks_Calendar.csv", header=True, inferSchema=True)

## Step 7: Add Audit Columns

In [0]:
def add_audit(df, source):
    return df.withColumn("ingestion_timestamp", current_timestamp()) \
             .withColumn("source_file", lit(source)) \
             .withColumn("layer", lit("bronze"))

## Step 8: Apply Audit Columns

In [0]:
sales_bronze = add_audit(sales, "sales_combined")

customers_bronze = add_audit(customers, "customers")

products_bronze = add_audit(products, "products")

categories_bronze = add_audit(categories, "categories")

subcategories_bronze = add_audit(subcategories, "subcategories")

returns_bronze = add_audit(returns, "returns")

territories_bronze = add_audit(territories, "territories")

calendar_bronze = add_audit(calendar, "calendar")

## Step 9: Save as Delta Tables

In [0]:
sales_bronze.write.format("delta").mode("overwrite").save(bronze_path + "sales")

customers_bronze.write.format("delta").mode("overwrite").save(bronze_path + "customers")

products_bronze.write.format("delta").mode("overwrite").save(bronze_path + "products")

categories_bronze.write.format("delta").mode("overwrite").save(bronze_path + "categories")

subcategories_bronze.write.format("delta").mode("overwrite").save(bronze_path + "subcategories")

returns_bronze.write.format("delta").mode("overwrite").save(bronze_path + "returns")

territories_bronze.write.format("delta").mode("overwrite").save(bronze_path + "territories")

calendar_bronze.write.format("delta").mode("overwrite").save(bronze_path + "calendar")

## Step 10: Validation

In [0]:
sales_df = spark.read.format("delta").load(bronze_path + "sales")
customers_df = spark.read.format("delta").load(bronze_path + "customers")
products_df = spark.read.format("delta").load(bronze_path + "products")

print("Sales Count:", sales_df.count())
print("Customers Count:", customers_df.count())
print("Products Count:", products_df.count())